# Phase 4 — XGBoost

End-to-end XGBoost workflow:
1. **EDA** — feature distributions, target relationship, leakage sanity, univariate predictive power
2. Training (next session) — hyperparameter from `src.models.config`, val early stopping, save artifacts
3. SHAP analysis (next session) — global importance + dependence plots on top features

In [ ]:
%load_ext autoreload
%autoreload 2

import sys; sys.path.insert(0, "..")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score

from src.models.data_loaders import load_xgb_split, class_imbalance_ratio

sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.max_rows", 80)

## 1. Exploratory Data Analysis

EDA is done on **train split only** — val/test stay sealed for honest evaluation.

### 1.1 Load and schema check

In [ ]:
X_train, y_train = load_xgb_split("train")
print(f"Shape: X={X_train.shape}  y={y_train.shape}")
print(f"Churn rate: {y_train.mean():.1%}  ({int(y_train.sum()):,} / {len(y_train):,})")
print(f"Class imbalance ratio (neg/pos): {class_imbalance_ratio(y_train):.2f}")
print(f"Feature dtypes:")
print(X_train.dtypes.value_counts())

### 1.2 Per-persona churn (sanity reminder)

Cross-check with the data-generation design: only `about_to_churn` and a sliver of `casual` should be churners. If we see hardcore/regular at non-zero, something drifted.

In [ ]:
users = pd.read_parquet("../data/raw/users.parquet", columns=["user_id","persona"])
train_personas = X_train.join(users.set_index("user_id"), how="left")
churn_by_persona = train_personas.assign(churn=y_train.values).groupby("persona")["churn"].agg(["sum","count","mean"])
print(churn_by_persona)

### 1.3 Missingness inventory

XGBoost handles NaN natively (it learns a default direction at each split). We still want to know which features are sparse — high NaN rate means low signal.

In [ ]:
null_count = X_train.isnull().sum()
null_pct = (null_count / len(X_train) * 100).round(1)
null_df = pd.DataFrame({"nulls": null_count, "pct": null_pct})
null_df = null_df[null_df["nulls"] > 0].sort_values("pct", ascending=False)
print(null_df.to_string())

### 1.4 Feature mean by churn class

Big mean shift = strong univariate signal. Look for ratios like `last_week_sessions: 9.7 → 1.4` (factor of 7x).

In [ ]:
class_means = X_train.assign(churn=y_train.values).groupby("churn").mean().T
class_means.columns = ["non_churn_mean", "churn_mean"]
class_means["ratio"] = class_means["churn_mean"] / class_means["non_churn_mean"].replace(0, np.nan)
class_means["abs_log_ratio"] = np.abs(np.log(class_means["ratio"].clip(lower=1e-6)))
# Top features by separation magnitude (ignoring NaN ratios from zero baselines)
class_means_sorted = class_means.sort_values("abs_log_ratio", ascending=False, na_position="last")
print(class_means_sorted.head(20).drop(columns="abs_log_ratio"))

### 1.5 Univariate predictive power (AUC per feature)

For each feature, train a one-feature classifier (just rank by the feature value) and compute ROC-AUC against churn. Higher = more individually informative. This is more robust than linear correlation for non-monotonic features.

`roc_auc_score(y, x)` returns 0.5 for "no signal" — flip a feature with AUC < 0.5 and you get AUC > 0.5, so what matters is `|AUC − 0.5|`.

In [ ]:
def univariate_auc(X, y):
    rows = []
    for col in X.columns:
        x = X[col].copy()
        if x.isnull().any():
            # Fill NaN with median for AUC computation; XGBoost handles NaN
            # natively at train time so this is only for the EDA ranking.
            x = x.fillna(x.median())
        if x.nunique() < 2:
            rows.append((col, np.nan, np.nan))
            continue
        auc = roc_auc_score(y, x)
        rows.append((col, auc, abs(auc - 0.5)))
    return pd.DataFrame(rows, columns=["feature","auc","abs_lift"]).sort_values("abs_lift", ascending=False).reset_index(drop=True)

ua = univariate_auc(X_train, y_train)
print("Top 15 features by univariate |AUC - 0.5|:")
print(ua.head(15).to_string(index=False))
print()
print("Bottom 10 (weakest univariate signal):")
print(ua.tail(10).to_string(index=False))

### 1.6 Leakage sanity

A single feature AUC ≥ 0.99 usually means leakage. None of our features should be that strong — they describe behavior *before* the prediction window, so they correlate but cannot perfectly predict.

In [ ]:
suspicious = ua[ua["auc"].notna() & (ua["abs_lift"] > 0.48)]
if len(suspicious) > 0:
    print("WARN: feature(s) with |AUC - 0.5| > 0.48 (suspect leakage):")
    print(suspicious.to_string(index=False))
else:
    print("PASS: no single feature is suspiciously predictive (max abs_lift = {:.3f})".format(ua["abs_lift"].max()))

### 1.7 Distribution plots for top 6 features

Visual check that the churn vs non-churn split is what we expect (decay features should show clear separation).

In [ ]:
top6 = ua.head(6)["feature"].tolist()
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, feat in zip(axes.ravel(), top6):
    df_plot = X_train[[feat]].assign(churn=y_train.values).dropna()
    for c, label, color in [(0, "non-churn", "#3b82f6"), (1, "churn", "#ef4444")]:
        sns.kdeplot(
            df_plot.loc[df_plot["churn"] == c, feat],
            label=label, color=color, fill=True, alpha=0.4, ax=ax,
        )
    ax.set_title(f"{feat}  (AUC={ua.loc[ua.feature==feat,'auc'].iloc[0]:.3f})")
    ax.legend()
plt.tight_layout()
plt.show()

### 1.8 Feature inter-correlation

Trees are robust to correlated features (unlike linear models) but heavy correlation can dilute SHAP attribution. We just want to flag any pair > 0.95 in case we want to drop one later.

In [ ]:
# Drop rows with NaN for the correlation matrix (per-pair pairwise would be cleaner)
corr = X_train.dropna().corr()
# Upper triangle, exclude diagonal
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
pairs = (
    corr.where(mask).stack().reset_index()
    .rename(columns={"level_0":"f1","level_1":"f2",0:"corr"})
)
high_corr = pairs[pairs["corr"].abs() > 0.85].sort_values("corr", key=abs, ascending=False)
print(f"Feature pairs with |corr| > 0.85: {len(high_corr)}")
print(high_corr.to_string(index=False))

### 1.9 EDA summary

Expected findings (data design):
- Top univariate features should be variants of `weekly_session_count_last`, `total_playtime_min_last`, `playtime_slope`, `longest_inactive_days_last`, `crash_exit_ratio_last`.
- No single feature should have AUC ≥ 0.99 (no leakage).
- Strong correlation between `weekly_session_count` and `total_playtime_min` (and their `_norm` / `_last` variants) is expected — they're all measuring the same underlying behavior.
- `peak_hour_ratio`, `weekend_ratio`, `genre_entropy` should have low univariate AUC (timing / diversity is a weaker churn signal than volume/decay).

## 2. Training

Train XGBoost on the train split with val-based early stopping. Hyperparameters
come from `src.models.config.XGB_PARAMS`. `scale_pos_weight` is computed
dynamically from the training labels (≈ 7.18 here).

The model picks its own `n_estimators` via early stopping (`patience=30` on
val PR-AUC) so we don't over- or under-train. Final iteration count is
reported as `best_iteration`.

### 2.1 Load data + train

In [ ]:
from src.models.xgboost_model import train_xgboost, predict_proba, save_artifacts
from src.models.data_loaders import load_xgb_split
from src.models.evaluate import compute_metrics, format_metrics

X_train, y_train = load_xgb_split("train")
X_val,   y_val   = load_xgb_split("val")
X_test,  y_test  = load_xgb_split("test")

clf, fit_info = train_xgboost(X_train, y_train, X_val, y_val)
print(f"best_iteration: {fit_info['best_iteration']}  (n_trees={fit_info['n_estimators_used']})")
print(f"scale_pos_weight: {fit_info['scale_pos_weight']:.3f}")
print(f"val: {format_metrics(fit_info['val_metrics'])}")

### 2.2 Training curve

Plot val PR-AUC across boosting rounds. Should rise sharply then plateau —
the early-stopping kicks in where the plateau hits its noise floor.

In [ ]:
results = clf.evals_result()
val_aucpr = results["validation_0"]["aucpr"]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(val_aucpr, color="#3b82f6")
ax.axvline(fit_info["best_iteration"], color="#ef4444", linestyle="--", label=f"best @ {fit_info['best_iteration']}")
ax.set_xlabel("boosting round")
ax.set_ylabel("val PR-AUC")
ax.set_title("XGBoost training curve")
ax.legend()
plt.tight_layout()
plt.show()

### 2.3 Predict on test + final metrics

Threshold for F1/precision/recall is selected on **val** and frozen for test.
Test ROC-AUC and PR-AUC are threshold-free.

In [ ]:
val_proba  = predict_proba(clf, X_val)
test_proba = predict_proba(clf, X_test)

artifacts = save_artifacts(
    clf,
    name="xgboost_v1",
    val_predictions=(y_val, val_proba),
    test_predictions=(y_test, test_proba),
    fit_info=fit_info,
)
print("val :", format_metrics(artifacts["val"]))
print("test:", format_metrics(artifacts["test"]))
print()
print("Saved:")
for k, v in artifacts["paths"].items():
    print(f"  {k:<13} {v}")

### 2.4 Confusion matrix

At the val-selected threshold, on test split.

In [ ]:
cm = artifacts["test"]["confusion_matrix"]
cm_arr = np.array([[cm["tn"], cm["fp"]], [cm["fn"], cm["tp"]]])

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(
    cm_arr, annot=True, fmt="d", cmap="Blues",
    xticklabels=["pred non-churn", "pred churn"],
    yticklabels=["true non-churn", "true churn"],
    cbar=False, ax=ax,
)
ax.set_title(f"XGBoost test set @ threshold={artifacts['test']['threshold']:.2f}")
plt.tight_layout()
plt.show()

# Quick business read
tp = cm["tp"]; fp = cm["fp"]; fn = cm["fn"]; tn = cm["tn"]
print(f"Caught {tp:,} of {tp+fn:,} actual churners (recall {artifacts['test']['recall']:.1%})")
print(f"Of {tp+fp:,} flagged users, {tp:,} actually churn (precision {artifacts['test']['precision']:.1%})")